In [1]:
import pyterrier as pt
import ir_datasets


dataset = ir_datasets.load("argsme/2020-04-01/touche-2020-task-1")

def find_doc(docid):
    for doc in dataset.docs_iter():
        # print(doc)
        # break
        if doc.doc_id == docid:
            return doc
        
find_doc("Sf9294c83-Af186e851")

KeyboardInterrupt: 

In [2]:
import json
import pandas as pd

lines = []
with open(r'datasets/kid-friend-en/docs.jsonl') as f:
    lines = f.read().splitlines()

line_dicts = [json.loads(line) for line in lines]
df_final = pd.DataFrame(line_dicts)
df_final.drop_duplicates(inplace=True,ignore_index=True)
df_final.head(1)

,docno,snippet,title,main_content
0,6e421f1539b1457b853712d81be87743,WEBBTS (also Bangtan Boys; Korean: 방탄소년단 Bangt...,BTS (band) - Wikipedia,BTS (also Bangtan Boys; Korean: 방탄소년단 Bangtan ...


In [ ]:
print("Number of documents: ", len(df_final))
print("Longest document length (in characters): ", max([len(text) for text in df_final["main_content"]]))
print("Longest length of document title (in characters): ", max([len(title) for title in df_final["title"]]))
print("Maximum length of doc id (in characters): ", max([len(docno) for docno in df_final["docno"]]))

Number of documents:  2385
Longest document length (in characters):  117554
Longest length of document title (in characters):  452
Maximum length of doc id (in characters):  32


In [ ]:
from pathlib import Path
import pyterrier as pt
import pandas as pd

# df_final.to_dict(orient="records")

idx_path = Path.cwd() / "index" / "kid-friend-en"
indexer = pt.IterDictIndexer(
    index_path = str(idx_path),
    meta={ # metadata recorded in index
        "docno": max([len(docno) for docno in df_final["docno"]]),
        "title": max([len(title) for title in df_final["title"]]),
        "snippet": max([len(snippet) for snippet in df_final["snippet"]]),
        "text": max([len(main_content) for main_content in df_final["main_content"]])
    },
    text_attrs = ["text"], # columns indexed
    stemmer="porter",
    stopwords="terrier",
)

Java started (triggered by TerrierIndexer.__init__) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [ ]:
def iter():
    for i, row in df_final.iterrows():
        yield {
            "docno": row["docno"],
            "title": row["title"],
            "snippet": row["snippet"],
            "text": row["main_content"]
        }

In [ ]:
index_ref = indexer.index(iter())

08:53:23.014 [main] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (37d53c33110a4231807e71e377f08377) - further warnings are suppressed
08:53:39.246 [main] WARN org.terrier.structures.indexing.Indexer -- Indexed 187 empty documents


In [ ]:
len(df_final.loc[df_final["main_content"]==""])

187

In [ ]:
idx_path = Path.cwd() / "index" / "kid-friend-en"
index = pt.IndexFactory.of(str(idx_path))

In [ ]:
tf_idf = pt.terrier.Retriever(index, wmodel="TF_IDF")

In [ ]:
page = 1
rpp = 10

items = tf_idf.search("bts")['docno'][page*rpp:(page+1)*rpp].tolist()
for i in items: 
    item = df_final.loc[df_final["docno"]==i]
    print({
        'title': item["title"].values[0],
        'snippet': item["snippet"].values[0]
    })

{'title': 'Which Bts Member Is Your Bias Quizzes | Quotev', 'snippet': 'Answer a set of questions to see which BTS member you are most like! Im sorry if this isnt your bias, this is just a comparison. ♡ Share your results!'}
{'title': 'BTS (band) - Wikipedia', 'snippet': 'BTS (also Bangtan Boys; Korean: 방탄소년단 Bangtan Sonyeondan) is a South Korean boy band of the third K-pop generation, consisting of seven ...'}
{'title': 'BTS | Spotify', 'snippet': 'Listen to BTS on Spotify. Artist - 26.6M monthly listeners. Preview of Spotify. Sign up to get unlimited songs and podcasts with occasional ads.'}
{'title': 'BTS - Wikipedia', 'snippet': 'BTS (Korean: 방탄소년단; RR: Bangtan Sonyeondan; lit. Bulletproof Boy Scouts), also known as the Bangtan Boys, is a South Korean boy band formed in 2010. The band consists of Jin, Suga, J-Hope, RM, Jimin, V, and Jungkook, who co-write or co-produce much of their material.Originally a hip hop group, they expanded their musical style to incorporate a wide range o

In [28]:
import pandas as pd
import re

commonlit = pd.read_excel("datasets/CLEAR Corpus 6.01.xlsx")
commonlit = commonlit[["ID", "Title", "Excerpt", "Author", "URL", "Source"]]

commonlit["Title"] = commonlit["Title"].apply(lambda x: re.sub(r"(\n+\s*)|(\s+)", " ", x.strip()))
commonlit["Excerpt"] = commonlit["Excerpt"].apply(lambda x: re.sub(r"(\n+\s*)|(\s+)", " ", x.strip()))
commonlit["ID"] = commonlit["ID"].apply(lambda x: str(x))

commonlit = commonlit.rename({"Title": "title", "Excerpt": "snippet", "Source": "source", "Author": "author", "ID": "docno", "URL": "url"}, axis=1)

In [29]:
commonlit.head()

,docno,title,snippet,author,url,source
0,400,Patty's Suitors,When the young people returned to the ballroom...,Carolyn Wells,http://www.gutenberg.org/cache/epub/5631/pg563...,gutenberg
1,401,Two Little Women on a Holiday,"All through dinner time, Mrs. Fayre was somewh...",Carolyn Wells,http://www.gutenberg.org/cache/epub/5893/pg589...,gutenberg
2,402,Patty Blossom,"As Roger had predicted, the snow departed as q...",Carolyn Wells,http://www.gutenberg.org/cache/epub/20945/pg20...,gutenberg
3,403,THE WATER-BABIES A Fairy Tale for a Land-Baby,Mr. Grimes was to come up next morning to Sir ...,CHARLES KINGSLEY,http://www.gutenberg.org/files/25564/25564-h/2...,gutenberg
4,404,HOW THE ARGONAUTS WERE DRIVEN INTO THE UNKNOWN...,And outside before the palace a great garden w...,Charles Kingsley,http://www.gutenberg.org/files/677/677-h/677-h...,gutenberg


In [30]:
commonlit.to_json("datasets/commonlit/docs.jsonl", orient="records", lines=True)

In [31]:
import json
import pandas as pd

lines = []
with open(r'datasets/commonlit/docs.jsonl') as f:
    lines = f.read().splitlines()

line_dicts = [json.loads(line) for line in lines]
df_final = pd.DataFrame(line_dicts)
df_final.drop_duplicates(inplace=True,ignore_index=True)
df_final.head(1)

,docno,title,snippet,author,url,source
0,400,Patty's Suitors,When the young people returned to the ballroom...,Carolyn Wells,http://www.gutenberg.org/cache/epub/5631/pg563...,gutenberg


In [32]:
from pathlib import Path
import pyterrier as pt
import pandas as pd

# df_final.to_dict(orient="records")

idx_path = Path.cwd() / "index" / "commonlit"
indexer = pt.IterDictIndexer(
    index_path = str(idx_path),
    meta={ # metadata recorded in index
        "docno": max([len(docno) for docno in df_final["docno"]]),
        "title": max([len(title) for title in df_final["title"]]),
        "snippet": max([len(snippet) for snippet in df_final["snippet"]]),
        "author": max([len(author) for author in df_final["author"]]),
        "url": max([len(url) for url in df_final["url"]]),
        "source": max([len(source) for source in df_final["source"]])
    },
    text_attrs = ["snippet"], # columns indexed
    stemmer="porter",
    stopwords="terrier",
)

Java started (triggered by TerrierIndexer.__init__) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


In [33]:
def iter():
    for i, row in df_final.iterrows():
        yield {
            "docno": row["docno"],
            "title": row["title"],
            "snippet": row["snippet"],
            "author": row["author"],
            "url": row["url"],
            "source": row["source"]
        }

In [34]:
index_ref = indexer.index(iter())

In [35]:
idx_path = Path.cwd() / "index" / "commonlit"
index = pt.IndexFactory.of(str(idx_path))

In [36]:
tf_idf = pt.terrier.Retriever(index, wmodel="TF_IDF")
page = 1
rpp = 10

items = tf_idf.search("boat")['docno'][page*rpp:(page+1)*rpp].tolist()
for i in items: 
    item = df_final.loc[df_final["docno"]==i]
    print({
        'title': item["title"].values[0],
        'snippet': item["snippet"].values[0]
    })

{'title': 'DOWN THE RIVER AFTER THE BOY', 'snippet': 'So he got into the boat, and began to rock it. The boat got loose, and drifted down the river. Walter did not notice this until he was quite a distance from the shore; then, turning round, he saw what had happened. Every moment the current was carrying him further from home. Walter was not a timid boy, and, instead of crying, he began to reason in this way: "The boat does not leak. It is safe and sound. There are no waves to make me afraid. The wind does not blow. Here on a seat is a thick blanket. In this box is a loaf of bread and a knife. The water of the river is good to drink, and here is a tin mug. I think I will not cry, but hope for the best." So he sat down. He called to some people on the shore; but they did not hear him. He stood up, and waved his hat to a man in a passing boat, and cried, "Help, help!" But the man thought it was some little fellow making fun of him.'}
{'title': 'THE LITTLE SAILOR', 'snippet': 'We had a m